# Lung Cancer Classification

This notebook is intentionally data-dependent: it stops with an actionable error when the authorized CSV is unavailable. It does not contain fabricated metrics or predictions.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from lung_cancer import (
    FEATURE_COLUMNS,
    RANDOM_STATE,
    TARGET_COLUMN,
    load_artifact,
    load_dataset,
    save_artifact,
    split_dataset,
    validate_feature_values,
)

DATA_PATH = Path(os.environ.get("LUNG_CANCER_DATA", "data.csv")).expanduser()
if not DATA_PATH.is_absolute():
    DATA_PATH = (Path.cwd() / DATA_PATH).resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Provide the authorized CSV as data.csv "
        "or set LUNG_CANCER_DATA to its path; no training or metrics were produced."
    )


## Load and validate the data contract

Column names are normalized only; values are not imputed. Rejecting missing/unknown values avoids silently changing the experiment.

In [ ]:
dataset, X, y = load_dataset(DATA_PATH)
class_counts = y.value_counts().sort_index()
print(f"Loaded {len(dataset)} rows and {len(FEATURE_COLUMNS)} features from {DATA_PATH}")
print("Class counts:", class_counts.to_dict())


## Leakage-safe split

The test partition is held out once. Model comparison and diagnostics use only the training partition.


In [ ]:
X_train, X_test, y_train, y_test = split_dataset(
    X, y, test_size=0.20, random_state=RANDOM_STATE, cv_splits=5
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


## Exploratory correlation (training partition only)

This descriptive plot uses training rows only and is not used to select features or fit a model.


In [ ]:
training_for_diagnostics = pd.concat([X_train, y_train.rename(TARGET_COLUMN)], axis=1)
correlation = training_for_diagnostics.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cbar=True, square=True, annot=False, cmap="vlag", center=0)
plt.title("Training feature/target correlation (descriptive only)")
plt.tight_layout()
plt.show()


## Candidate models and cross-validation

Scaling is fitted inside a pipeline, separately within each training fold. The test set is not used for model selection.

In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN classifier": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
    "Decision tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=200, criterion="entropy", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for model_name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, cv=cv, scoring="f1", error_score="raise"
    )
    cv_rows.append({
        "model": model_name,
        "cv_f1_mean": scores.mean(),
        "cv_f1_std": scores.std(),
    })
cv_results = pd.DataFrame(cv_rows).sort_values(
    by=["cv_f1_mean", "model"], ascending=[False, True], ignore_index=True
)
cv_results


## Final held-out evaluation

The best model is selected by training-fold CV only, then refit on all training rows. Metrics below are produced from the untouched test partition; they are not hard-coded.

In [ ]:
best_model_name = cv_results.loc[0, "model"]
classifier = clone(models[best_model_name])
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

metrics = {
    "model": best_model_name,
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
}
if hasattr(classifier, "predict_proba"):
    metrics["roc_auc"] = roc_auc_score(y_test, classifier.predict_proba(X_test)[:, 1])
print(metrics)
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Held-out test confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## Save the trained model

The artifact is generated only after a successful data-backed run and is ignored by Git.

In [ ]:
ARTIFACT_PATH = Path(os.environ.get("LUNG_CANCER_ARTIFACT", "artifacts/lc.pkl")).expanduser()
if not ARTIFACT_PATH.is_absolute():
    ARTIFACT_PATH = (Path.cwd() / ARTIFACT_PATH).resolve()
MODEL_PATH = save_artifact(classifier, ARTIFACT_PATH, random_state=RANDOM_STATE)
# Reload the persisted envelope so schema checks also cover the artifact-loading path.
artifact = load_artifact(MODEL_PATH)
classifier = artifact["model"]
print(f"Saved and reloaded model artifact at {MODEL_PATH}")


## Prediction API (requires caller-provided values)

No arbitrary sample row is embedded. Pass a mapping with the exact feature names and use the same encodings as the validated CSV (`GENDER`: `M`/`F`, numeric risk factors).

In [ ]:
def predict_one(feature_values: dict[str, object]) -> int:
    """Predict one row after enforcing the exact training schema and finite values."""
    row = validate_feature_values(feature_values)
    return int(classifier.predict(row)[0])
